In [ ]:
import json
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

RESULTS_DIR = "E:/001NetOps/merged_results"
# set the max rows to display to 100
pl.Config.set_tbl_rows(50)

plt.rcParams.update(
    {
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "lines.linewidth": 1,
        "figure.dpi": 300,
    }
)

palette_1 = plt.get_cmap("Set1")([0, 1, 2])

In [ ]:
eval_results_df = pl.read_csv(f"{RESULTS_DIR}/0_summary/evaluation_summary.csv")
eval_results_df = eval_results_df.with_columns(
    pl.when(pl.col("scenario_topo_size") == "-")
    .then(pl.lit("s"))
    .otherwise(pl.col("scenario_topo_size"))
    .alias("scenario_topo_size")
)


In [ ]:
eval_results_df

In [ ]:
eval_results_df.group_by(["root_cause_name", "scenario_topo_size", "backend_model"]).agg(pl.len()).sort(
    by="len", descending=False
)

In [ ]:
# get reasoning tokens statistics
reasoning_tokens_df = []

for row in eval_results_df.iter_rows(named=True):
    dst_dir = os.path.join(RESULTS_DIR, row["root_cause_name"], str(row["session_id"]))
    conv_file = os.path.join(dst_dir, "conversation_diagnosis_agent.log")
    if not os.path.exists(conv_file):
        print(f"Conversation file not found: {conv_file}")
        print(row)
        continue

    with open(conv_file, "r") as f:
        tot_reas_session_tokens = 0
        for line in f:
            try:
                record = json.loads(line)
                if record["event"] == "llm_end":
                    try:
                        reas_tokens = record["usage_metadata"]["output_token_details"]["reasoning"]
                        tot_reas_session_tokens += reas_tokens
                    except KeyError:
                        pass
            except json.JSONDecodeError:
                print(f"JSON decode error in file: {conv_file}")
                print(f"Line: {line}")
                continue

        reasoning_tokens_df.append({"session_id": row["session_id"], "reasoning_tokens": tot_reas_session_tokens})
reasoning_tokens_df = pl.DataFrame(reasoning_tokens_df)
eval_results_df = eval_results_df.join(reasoning_tokens_df, on="session_id", how="left")

In [ ]:
eval_results_df.head()

In [ ]:
eval_results_df.filter(pl.col("backend_model") == "gpt-5").group_by(["net_env", "root_cause_name"]).agg(
    pl.len()
).select(pl.col("len").sum() / 2)

In [ ]:
eval_results_df.group_by("backend_model").agg(
    pl.len().alias("num_sessions"),
    (pl.col("time_taken").sum() / 60 / 60).round(1).alias("total_hours"),
    (pl.col("time_taken").mean() / 60 / 60).round(1).alias("average_hours"),
).sort("total_hours", descending=True)

In [ ]:
# # fix ground truth
# def get_metrics(sub_list, gt_list):
#     tp = len(set(sub_list) & set(gt_list))
#     fn = len(set(gt_list) - set(sub_list))
#     fp = len(set(sub_list) - set(gt_list))

#     precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
#     recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
#     accuracy = tp / len(gt_list) if len(gt_list) > 0 else 0.0

#     if precision + recall == 0:
#         f1 = 0.0
#     else:
#         f1 = 2 * precision * recall / (precision + recall)
#     return {
#         "precision": round(precision, 4),
#         "recall": round(recall, 4),
#         "f1": round(f1, 4),
#         "accuracy": round(accuracy, 4),
#     }


# new_eval_results = []

# for row in eval_results_df.iter_rows(named=True):
#     dst_dir = os.path.join(RESULTS_DIR, row["root_cause_name"], str(row["session_id"]))
#     sub_file = os.path.join(dst_dir, "submission.json")
#     if not os.path.exists(sub_file):
#         new_eval_results.append(row)
#         continue

#     with open(sub_file, "r") as f:
#         submission = json.load(f)
#     with open(os.path.join(dst_dir, "ground_truth.json"), "r") as f:
#         ground_truth = json.load(f)

#     detection_score = 1 if submission["is_anomaly"] == ground_truth["is_anomaly"] else 0
#     metrics = {
#         "localization": get_metrics(submission["faulty_devices"], ground_truth["faulty_devices"]),
#         "rca": get_metrics(submission["root_cause_name"], ground_truth["root_cause_name"]),
#     }

#     # check if the scores are different from the eval_results_df
#     for task in ["localization", "rca"]:
#         for m in ["accuracy", "precision", "recall", "f1"]:
#             if row[f"{task}_{m}"] != metrics[task][m]:
#                 print(
#                     f"Mismatch in {task} {m} for session {row['session_id']} problem {row['root_cause_name']}: eval_results_df={row[f'{task}_{m}']}, computed={metrics[task][m]}"
#                 )
#                 row[f"{task}_{m}"] = metrics[task][m]
#     new_eval_results.append(row)
# new_eval_results_df = pl.DataFrame(new_eval_results)
# new_eval_results_df

# reasoning steps v.s. success rate

no obvious pattern

In [ ]:
# import seaborn as sns

# model = "gpt-5"
# plt_df = eval_results_df.with_columns(
#     (( pl.col("localization_accuracy"))).alias("total_score")
# )#.filter(pl.col("backend_model") == model)

# # scatter plot
# plt.figure(figsize=(4, 3))
# sns.scatterplot(
#     data=plt_df.to_pandas(),
#     x="reasoning_tokens",
#     y="total_score",
#     hue="scenario_topo_size",
#     palette="Set1",
#     s=10,
# )
# plt.xlabel("Reasoning tokens")
# # plt.xscale("log")
# plt.show()

# LLMs

In [ ]:
# replace -1 with 0
for col in ["detection_score", "localization_accuracy", "rca_accuracy"]:
    eval_results_df = eval_results_df.with_columns(pl.when(pl.col(col) < 0).then(0).otherwise(pl.col(col)).alias(col))
res = (
    eval_results_df.group_by(["backend_model"])
    .agg(
        [
            (pl.col("time_taken").mean()).round(1).alias("mean_time_taken"),
            pl.col("steps").mean().round(1).alias("mean_steps"),
            pl.col("tool_calls").mean().round(1).alias("mean_tool_calls"),
            # pl.col("tool_errors").mean().round(2).alias("mean_tool_errors"),
            pl.col("in_tokens").mean().round(1).alias("mean_in_tokens"),
            pl.col("out_tokens").mean().round(1).alias("mean_out_tokens"),
            # pl.col("reasoning_tokens").mean().round(1).alias("mean_reasoning_tokens"),
            ((pl.col("detection_score").mean() * 100).round(1).cast(pl.Utf8) + "\\%").alias("mean_detection_accuracy"),
            ((pl.col("localization_accuracy").mean() * 100).round(1).cast(pl.Utf8) + "\\%").alias(
                "mean_localization_accuracy"
            ),
            ((pl.col("rca_accuracy").mean() * 100).round(1).cast(pl.Utf8) + "\\%").alias("mean_rca_accuracy"),
        ]
    )
    .sort(["backend_model"], descending=True)
)


display(res)

print(res.to_pandas().to_csv(index=False).replace(",", " & ").replace("\n", " \\\\ \n"))

# Category

In [ ]:
res = (
    eval_results_df.filter(pl.col("detection_score") != -1)
    .group_by(["backend_model", "root_cause_category"])
    .agg(
        [
            pl.len().alias("num_cases"),
            pl.col("time_taken").mean().round(1).alias("mean_time_taken"),
            pl.col("steps").mean().round(1).alias("mean_steps"),
            pl.col("tool_calls").mean().round(1).alias("mean_tool_calls"),
            ((pl.col("in_tokens") + pl.col("out_tokens")).mean() / 1000).round(1).alias("mean_total_tokens"),
            ((pl.col("detection_score").mean() * 100).round(1)).alias("mean_detection_accuracy"),
            ((pl.col("localization_accuracy").mean() * 100).round(1)).alias("mean_localization_accuracy"),
            ((pl.col("rca_accuracy").mean() * 100).round(1)).alias("mean_rca_accuracy"),
        ]
    )
    .sort(["mean_detection_accuracy"], descending=True)
)
# res['mean_detection_accuracy'].mean()
# res = res.with_columns(
#     (pl.col("root_cause_category").str.replace_all("_", "\\_")),
#     (pl.col("mean_detection_accuracy").cast(pl.Utf8) + "\\%").alias("mean_detection_accuracy"),
#     (pl.col("mean_localization_accuracy").cast(pl.Utf8) + "\\%").alias("mean_localization_accuracy"),
#     (pl.col("mean_rca_accuracy").cast(pl.Utf8) + "\\%").alias("mean_rca_accuracy"),
# )

res = res.with_columns(
    pl.col("root_cause_category")
    .str.replace("link_failure", "LF")
    .replace("network_node_error", "NE")
    .replace("network_under_attack", "NA")
    .replace("end_host_failure", "HF")
    .replace("misconfiguration", "MC")
    .replace("resource_contention", "RC"),
)

display(res)
print(res.to_pandas().to_csv(index=False).replace(",", " & ").replace("\n", " \\\\ \n"))

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

mpl.rcParams.update(
    {
        "hatch.linewidth": 0.5,
        "axes.linewidth": 0.6,
    }
)

metrics = [
    "mean_detection_accuracy",
    "mean_localization_accuracy",
    "mean_rca_accuracy",
]
names = ["Accuracy (%)", "Accuracy (%)", "Accuracy (%)"]


metric_colors = [
    "#2c7fb8",  # steel blue
    "#41b6c4",  # cyan-blue
    "#253494",  # navy-indigo
]


model_a = "gpt-oss:20b"
model_b = "gpt-5-mini"
model_c = "gpt-5"

for col, ylabel, base_color in zip(metrics, names, metric_colors):
    plt_df = res.select(["backend_model", "root_cause_category", col]).to_pandas()
    pivot = plt_df.pivot(index="root_cause_category", columns="backend_model", values=col).sort_index()

    c_deep, c_mid, c_light = sns.light_palette(base_color, n_colors=3, reverse=True)

    model_style = {
        model_a: dict(color=c_deep, hatch=None, alpha=1.0, linewidth=0.4),
        model_b: dict(color=c_mid, hatch="---", alpha=0.95, linewidth=0.4),
        model_c: dict(color=c_light, hatch="///", alpha=0.90, linewidth=0.4),
    }

    cats = pivot.index.to_list()
    x = np.arange(len(cats))
    width = 0.22

    fig, ax = plt.subplots(figsize=(2.1, 1.0), constrained_layout=True)

    for i, (cat, x_pos) in enumerate(zip(cats, x)):
        vals = {
            model_a: pivot.loc[cat, model_a],
            model_b: pivot.loc[cat, model_b],
            model_c: pivot.loc[cat, model_c],
        }

        for offset, model in zip([-width, 0.0, width], [model_a, model_b, model_c]):
            st = model_style[model]
            ax.bar(
                x_pos + offset,
                vals[model],
                width=width,
                label=model if i == 0 else None,
                color=st["color"],
                edgecolor="black",
                hatch=st["hatch"],
                alpha=st["alpha"],
                linewidth=st["linewidth"],
            )

    if "Acc" in ylabel:
        ax.set_ylim(0, 100)

    ax.set_ylabel(ylabel, fontsize=8, loc="top")
    ax.set_xticks(x)
    ax.set_xticklabels(cats, fontsize=8)

    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.3)
    ax.yaxis.set_major_locator(MaxNLocator(nbins="auto", min_n_ticks=6))

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    # ax.legend(frameon=False, fontsize=6, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.25))

    fig.savefig(
        f"figs/issue_category_{col}.pdf",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.01,
        transparent=True,
    )

    plt.show()


In [ ]:
res = (
    eval_results_df.filter(pl.col("detection_score") != -1)
    .group_by(["root_cause_name"])
    .agg(
        [
            pl.len().alias("num_cases"),
            pl.col("time_taken").mean().round(1).alias("mean_time_taken"),
            pl.col("steps").mean().round(1).alias("mean_steps"),
            pl.col("tool_calls").mean().round(1).alias("mean_tool_calls"),
            (pl.col("in_tokens") + pl.col("out_tokens")).mean().round(1).alias("mean_total_tokens"),
            ((pl.col("detection_score").mean() * 100).round(1)).alias("mean_detection_accuracy"),
            ((pl.col("localization_accuracy").mean() * 100).round(1)).alias("mean_localization_accuracy"),
            ((pl.col("rca_accuracy").mean() * 100).round(1)).alias("mean_rca_accuracy"),
        ]
    )
    .sort(["mean_rca_accuracy"], descending=False)
)

display(res)

# size

In [ ]:
from llm4netlab.net_env.net_env_pool import _NET_ENVS

machines_count = {"s": [], "m": [], "l": []}
for net_env in _NET_ENVS.values():
    if net_env.TOPO_SIZE:
        for size in ["s", "m", "l"]:
            machines_count[size].append(len(net_env(topo_size=size).lab.machines))

machines_count


In [ ]:
for size, counts in machines_count.items():
    print(f"Average machines count for size {size}: {np.mean(counts)}")

In [ ]:
eval_results_df.group_by("scenario_topo_size").agg(pl.col("time_taken").sum())

In [ ]:
eval_results_df.select([pl.col("net_env"), pl.col("scenario_topo_size")]).unique().sort(
    ["net_env", "scenario_topo_size"]
)

In [ ]:
from copy import deepcopy

tmp_df = deepcopy(eval_results_df)

for col in ["detection_score", "localization_accuracy", "rca_accuracy"]:
    tmp_df = tmp_df.with_columns(pl.when(pl.col(col) == -1).then(pl.lit(0)).otherwise(pl.col(col)).alias(col))

res = (
    tmp_df.group_by("backend_model", "scenario_topo_size")
    .agg(
        [
            pl.len().alias("num_cases"),
            pl.col("time_taken").mean().round(1).alias("mean_time_taken"),
            pl.col("steps").mean().round(1).alias("mean_steps"),
            pl.col("tool_calls").mean().round(1).alias("mean_tool_calls"),
            ((pl.col("in_tokens") + pl.col("out_tokens")).mean() / 1000).round(2).alias("mean_total_tokens"),
            (pl.col("detection_score").mean() * 100).round(1).alias("mean_detection_accuracy"),
            (pl.col("localization_accuracy").mean() * 100).round(1).alias("mean_localization_accuracy"),
            (pl.col("rca_accuracy").mean() * 100).round(1).alias("mean_rca_accuracy"),
        ]
    )
    .sort("scenario_topo_size", descending=True)
)
res = res.with_columns(
    pl.col("scenario_topo_size").str.replace("s", "S").replace("m", "M").replace("l", "L"),
)

display(res)
print(res.to_pandas().to_csv(index=False).replace(",", " & ").replace("\n", " \\\\ \n"))

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import MaxNLocator

mpl.rcParams["hatch.linewidth"] = 0.5

metrics = [
    "mean_detection_accuracy",
    "mean_localization_accuracy",
    "mean_rca_accuracy",
    "mean_tool_calls",
    "mean_total_tokens",
]
names = ["Accuracy (%)", "Accuracy (%)", "Accuracy (%)", "# Tool Calls", "# Tokens (k)"]

metric_colors = [
    "#2c7fb8",  # steel blue
    "#41b6c4",  # cyan-blue
    "#253494",  # navy-indigo
    sns.color_palette("tab10")[1],  # tool calls
    sns.color_palette("tab10")[2],  # tokens
]

model_a = "gpt-oss:20b"
model_b = "gpt-5-mini"
model_c = "gpt-5"

for col, ylabel, base_color in zip(metrics, names, metric_colors):
    plt_df = res.select(["backend_model", "scenario_topo_size", col]).to_pandas()

    pivot = plt_df.pivot(index="scenario_topo_size", columns="backend_model", values=col).reindex(index=["S", "M", "L"])

    c_deep, c_mid, c_light = sns.light_palette(base_color, n_colors=3, reverse=True)

    model_style = {
        model_a: dict(color=c_deep, hatch=None, linewidth=0.4),
        model_b: dict(color=c_mid, hatch="---", linewidth=0.4),
        model_c: dict(color=c_light, hatch="///", linewidth=0.4),
    }

    x = np.arange(len(pivot.index))
    width = 0.2

    fig, ax = plt.subplots(figsize=(1.6, 1.0), constrained_layout=True)

    for i, (topo, x_pos) in enumerate(zip(pivot.index, x)):
        for offset, model in zip([-width, 0.0, width], [model_a, model_b, model_c]):
            st = model_style[model]
            ax.bar(
                x_pos + offset,
                pivot.loc[topo, model],
                width=width,
                label=model if i == 0 else None,
                color=st["color"],
                hatch=st["hatch"],
                edgecolor="black",
                linewidth=st["linewidth"],
            )

    if "Acc" in ylabel:
        ax.set_ylim(0, 100)

    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, fontsize=8)

    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.3)
    ax.yaxis.set_major_locator(MaxNLocator(nbins="auto", min_n_ticks=6))

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    # ax.legend(frameon=False, fontsize=6, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.25))

    fig.savefig(
        f"figs/size_{col}.pdf",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.01,
        transparent=True,
    )

    plt.show()


In [ ]:
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns

# hatch 粗细和正文一致
mpl.rcParams["hatch.linewidth"] = 0.5

# 用一个中性的蓝色做 legend（不绑定具体 metric）
base_blue = "#4c72b0"
c_deep, c_mid, c_light = sns.light_palette(base_blue, n_colors=3, reverse=True)

legend_patches = [
    mpatches.Patch(
        facecolor=c_deep,
        edgecolor="black",
        linewidth=0.4,
        label="GPT-OSS:20B",
    ),
    mpatches.Patch(
        facecolor=c_mid,
        edgecolor="black",
        hatch="---",
        linewidth=0.4,
        label="GPT-5-mini",
    ),
    mpatches.Patch(
        facecolor=c_light,
        edgecolor="black",
        hatch="///",
        linewidth=0.4,
        label="GPT-5",
    ),
]

fig, ax = plt.subplots(figsize=(2.2, 0.45))
ax.axis("off")

ax.legend(
    handles=legend_patches,
    loc="center",
    ncol=3,
    frameon=False,
    fontsize=8,
    handlelength=1.8,
    handleheight=0.9,
    columnspacing=1.2,
)

plt.tight_layout()
fig.savefig(
    "figs/model_legend.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)
plt.show()


# Task

In [ ]:
# study the acc, precsion and false positiv

In [ ]:
eval_results_df.filter(pl.col("detection_score") != -1).group_by(["backend_model", "root_cause_category"]).agg(
    [
        pl.len().alias("num_cases"),
        pl.col("detection_score").mean().alias("mean_detection_accuracy"),
        pl.col("localization_accuracy").mean().alias("mean_localization_accuracy"),
        pl.col("rca_accuracy").mean().alias("mean_rca_accuracy"),
    ]
).sort(["backend_model", "root_cause_category"], descending=True)

In [ ]:
# fix ground truth
def get_metrics(sub_list, gt_list):
    tp = len(set(sub_list) & set(gt_list))
    fn = len(set(gt_list) - set(sub_list))
    fp = len(set(sub_list) - set(gt_list))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = tp / len(gt_list) if len(gt_list) > 0 else 0.0

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "accuracy": round(accuracy, 4),
    }


new_eval_results = []

for row in eval_results_df.iter_rows(named=True):
    dst_dir = os.path.join(RESULTS_DIR, row["root_cause_name"], str(row["session_id"]))
    sub_file = os.path.join(dst_dir, "submission.json")
    if not os.path.exists(sub_file):
        new_eval_results.append(row)
        continue

    with open(sub_file, "r") as f:
        submission = json.load(f)
    with open(os.path.join(dst_dir, "ground_truth.json"), "r") as f:
        ground_truth = json.load(f)

    detection_score = 1 if submission["is_anomaly"] == ground_truth["is_anomaly"] else 0
    metrics = {
        "localization": get_metrics(submission["faulty_devices"], ground_truth["faulty_devices"]),
        "rca": get_metrics(submission["root_cause_name"], ground_truth["root_cause_name"]),
    }

    # check if the scores are different from the eval_results_df
    for task in ["localization", "rca"]:
        for m in ["accuracy", "precision", "recall", "f1"]:
            if row[f"{task}_{m}"] != metrics[task][m]:
                print(
                    f"Mismatch in {task} {m} for session {row['session_id']} problem {row['root_cause_name']}: eval_results_df={row[f'{task}_{m}']}, computed={metrics[task][m]}"
                )
                row[f"{task}_{m}"] = metrics[task][m]
    new_eval_results.append(row)
new_eval_results_df = pl.DataFrame(new_eval_results)
new_eval_results_df

# Tool invocation analysis

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np

llm_tool_usage_stats = {}

tool_abbr = {
    "ping_pair": "PING",
    "exec_shell": "EXEC",
    "get_host_net_config": "NETCFG",
    "frr_exec": "FRR-EX",
    "curl_web_test": "CURL",
    "frr_show_ip_route": "FRR-RT",
    "get_reachability": "REACH",
    "systemctl_ops": "SYSCTL",
    "netstat": "NETSTAT",
    "frr_show_running_config": "FRR-CFG",
    "frr_get_bgp_conf": "FRR-BGP",
    "frr_get_ospf_conf": "FRR-OSPF",
    "ethtool": "ETHTOOL",
    "get_tc_statistics": "TC-STAT",
    "ip_addr_statistics": "IP-STAT",
    "bmv2_get_log": "BMV2-LOG",
    "exec_shell_dual": "EXEC-DUAL",
}


for llm in ["gpt-5-mini", "gpt-5"]:
    detection_tool_usage = {"success": Counter(), "failure": Counter()}
    localization_tool_usage = {"success": Counter(), "failure": Counter()}
    rca_tool_usage = {"success": Counter(), "failure": Counter()}
    all_tool_usage = {"success": Counter(), "failure": Counter()}

    tmp_df = eval_results_df.filter(pl.col("backend_model") == llm)
    for row in tmp_df.iter_rows(named=True):
        dst_dir = os.path.join(RESULTS_DIR, row["root_cause_name"], str(row["session_id"]))
        conv_file = os.path.join(dst_dir, "conversation_diagnosis_agent.log")
        if not os.path.exists(conv_file):
            print(f"Conversation file not found: {conv_file}")
            print(row)
            continue

        with open(conv_file, "r") as f:
            for line in f:
                record = json.loads(line)
                if record["event"] == "tool_start":
                    tool = record["tool"]["name"]
                    tool = tool.replace("running_", "").replace("host_net_config", "net_cfg")
                    # if tool in tool_abbr:
                    #     tool = tool_abbr[tool]
                    # else:
                    #     print(f"Unknown tool name: {tool}")
                    #     break
                    if row["detection_score"] == 1:
                        detection_tool_usage["success"][tool] += 1
                    else:
                        detection_tool_usage["failure"][tool] += 1
                    if row["localization_accuracy"] > 0:
                        localization_tool_usage["success"][tool] += 1
                    else:
                        localization_tool_usage["failure"][tool] += 1
                    if row["rca_accuracy"] > 0:
                        rca_tool_usage["success"][tool] += 1
                    else:
                        rca_tool_usage["failure"][tool] += 1
                    # all success
                    if row["detection_score"] == 1 and row["localization_accuracy"] > 0 and row["rca_accuracy"] > 0:
                        all_tool_usage["success"][tool] += 1
                    else:
                        all_tool_usage["failure"][tool] += 1

                if record["event"] == "tool_error":
                    tool = record["output"]
                    match = re.search(r"name='([^']+)'", tool)
                    if match:
                        tool = match.group(1)

                    tool = "invalid_tool"
                    if row["detection_score"] == 1:
                        detection_tool_usage["success"][tool] += 1
                    else:
                        detection_tool_usage["failure"][tool] += 1
                    if row["localization_accuracy"] > 0:
                        localization_tool_usage["success"][tool] += 1
                    else:
                        localization_tool_usage["failure"][tool] += 1
                    if row["rca_accuracy"] > 0:
                        rca_tool_usage["success"][tool] += 1
                    else:
                        rca_tool_usage["failure"][tool] += 1
                    # all success
                    if row["detection_score"] == 1 and row["localization_accuracy"] > 0 and row["rca_accuracy"] > 0:
                        all_tool_usage["success"][tool] += 1
                    else:
                        all_tool_usage["failure"][tool] += 1

                    # count as failure for all
                    # detection_tool_usage["failure"][tool] += 1
                    # localization_tool_usage["failure"][tool] += 1
                    # rca_tool_usage["failure"][tool] += 1
                    # all_tool_usage["failure"][tool] += 1

    llm_tool_usage_stats[llm] = {
        "detection_tool_usage": detection_tool_usage,
        "localization_tool_usage": localization_tool_usage,
        "rca_tool_usage": rca_tool_usage,
        "all_tool_usage": all_tool_usage,
    }


model = "gpt-5-mini"
tool_usage = []

for tool in set(
    list(llm_tool_usage_stats[model]["detection_tool_usage"]["success"].keys())
    + list(llm_tool_usage_stats[model]["detection_tool_usage"]["failure"].keys())
):
    tmp_sample = {"tool": tool}
    for k in ["detection", "localization", "rca", "all"]:
        tmp_success_num = llm_tool_usage_stats[model][f"{k}_tool_usage"]["success"].total()
        tmp_failure_num = llm_tool_usage_stats[model][f"{k}_tool_usage"]["failure"].total()

        tmp_sample[f"{k}_success"] = llm_tool_usage_stats[model][f"{k}_tool_usage"]["success"][tool] / tmp_success_num
        tmp_sample[f"{k}_failure"] = llm_tool_usage_stats[model][f"{k}_tool_usage"]["failure"][tool] / tmp_failure_num

    tool_usage.append(tmp_sample)

tool_usage_df = pl.DataFrame(tool_usage)
tool_usage_df

In [ ]:
import seaborn as sns

# select the top10
plt_df = tool_usage_df.select(["tool", "all_success", "all_failure"]).sort("all_success", descending=True).head(10)

plt_df = pl.concat(
    [
        plt_df,
        tool_usage_df.select(["tool", "all_success", "all_failure"]).filter(pl.col("tool") == "invalid_tool"),
    ]
)

plt.figure(figsize=(5, 2), constrained_layout=True)
bar_width = 0.3
index = np.arange(len(plt_df))
palette_tab10 = sns.color_palette("tab10")
n = len(plt_df)
success_colors = [palette_tab10[0]] * (n - 1) + ["#d62728"]
failure_colors = [palette_tab10[1]] * (n - 1) + ["#d62728"]

# ---------- 左图：Success ----------
plt.subplot(1, 2, 1)
success_vals = plt_df["all_success"].to_list()
success_vals = [v * 100 for v in success_vals]
plt.barh(index, success_vals, height=bar_width, color=success_colors, label="Success cases")

# 标注数值
for i, v in enumerate(success_vals):
    plt.text(v + 0.005, i, f"{v:.1f}%", va="center", ha="left")

plt.xlabel("Tool invocation rate (%)")
# plt.title("(a) Success cases")
plt.legend(frameon=False, fontsize=7)
plt.xlim(0, max(success_vals) * 1.2)
plt.yticks(index + bar_width / 2, plt_df["tool"].to_list())
plt.grid(axis="x", linestyle="--", color="black", linewidth=0.5, alpha=0.3)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

# ---------- 右图：Failure ----------
plt.subplot(1, 2, 2)
failure_vals = plt_df["all_failure"].to_list()
failure_vals = [v * 100 for v in failure_vals]
plt.barh(index, failure_vals, height=bar_width, color=failure_colors, label="Failure cases")

# 标注数值
for i, v in enumerate(failure_vals):
    plt.text(v + 0.005, i, f"{v:.1f}%", va="center", ha="left")

plt.xlabel("Tool invocation rate (%)")
# plt.title("(b) Failure cases")
plt.legend(
    frameon=False,
    fontsize=7,
)
plt.yticks([])
plt.xlim(0, max(failure_vals) * 1.2)

plt.grid(axis="x", linestyle="--", color="black", linewidth=0.5, alpha=0.3)
# plt.tight_layout()
# plt.subplots_adjust(wspace=0.05)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)
plt.savefig(
    f"figs/tool_{model}.pdf",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.01,
    transparent=True,
)
plt.show()

## check shell tool invocation

In [ ]:
import ast
import json
from collections import Counter

cmd_exec_counter = {"gpt-5-mini": Counter(), "gpt-5": Counter()}

common_cmd_list = [
    "ping ",
    "traceroute ",
    "ip neigh",
    "dig ",
    "nslookup ",
    "curl ",
    "ip route",
    "ip link",
    "iptables ",
    "arping ",
    "ps ",
    "tcpdump ",
    "cat ",
    "ls ",
]

for row in eval_results_df.iter_rows(named=True):
    dst_dir = os.path.join(RESULTS_DIR, row["root_cause_name"], str(row["session_id"]))
    conv_file = os.path.join(dst_dir, "conversation_diagnosis_agent.log")
    if not os.path.exists(conv_file):
        print(f"Conversation file not found: {conv_file}")
        print(row)
        continue

    with open(conv_file, "r") as f:
        for line in f:
            record = json.loads(line)
            if record["event"] == "tool_start":
                if record["tool"]["name"] == "exec_shell":
                    cmd = ast.literal_eval(record["input"])
                    if "command" in cmd:
                        cmd = cmd["command"]
                        for common_cmd in common_cmd_list:
                            if common_cmd in cmd:
                                cmd_exec_counter[row["backend_model"]][common_cmd.strip()] += 1

cmd_exec_counter

In [ ]:
cmd_exec_counter = {"gpt-5-mini": Counter(), "gpt-5": Counter()}

common_cmd_list = [
    "ping ",
    "traceroute ",
    "ip neigh",
    "dig ",
    "nslookup ",
    "curl ",
    "ip route",
    "ip link",
    "iptables ",
    "arping ",
    "ps ",
    "tcpdump ",
    "cat ",
    "ls ",
]

for row in eval_results_df.iter_rows(named=True):
    dst_dir = os.path.join(RESULTS_DIR, row["root_cause_name"], str(row["session_id"]))
    conv_file = os.path.join(dst_dir, "conversation_diagnosis_agent.log")
    if not os.path.exists(conv_file):
        print(f"Conversation file not found: {conv_file}")
        print(row)
        continue

    with open(conv_file, "r") as f:
        for line in f:
            record = json.loads(line)
            if record["event"] == "tool_start":
                if record["tool"]["name"] == "exec_shell_dual":
                    cmd = ast.literal_eval(record["input"])
                    print(cmd)

## tool errors

In [ ]:
import json
from collections import Counter

tool_count = {
    "gpt-5-mini": Counter({"success": 0, "error": 0}),
    "gpt-5": Counter({"success": 0, "error": 0}),
}

for row in eval_results_df.iter_rows(named=True):
    dst_dir = os.path.join(RESULTS_DIR, row["root_cause_name"], str(row["session_id"]))
    conv_file = os.path.join(dst_dir, "conversation_diagnosis_agent.log")
    if not os.path.exists(conv_file):
        print(f"Conversation file not found: {conv_file}")
        print(row)
        continue

    with open(conv_file, "r") as f:
        for line in f:
            record = json.loads(line)
            if record["event"] == "tool_error":
                # print(f"Tool error in session {row['session_id']}, {row['root_cause_name']}, {record['output']}")
                tool_count[row["backend_model"]]["error"] += 1
            if record["event"] == "tool_start":
                tool_count[row["backend_model"]]["success"] += 1
print(tool_count)

## tool by different LLMs